# Redrob Candidate Ranker — Sandbox Demo

This notebook is the **sandbox link** required by the Redrob hackathon submission spec (Section 10.5). It runs the full ranking pipeline end-to-end on a small sample of candidates so the architecture can be verified without needing the full 100K-candidate pool or a local environment.

Full code, the complete rubric, and the real 100K-scale output live in the GitHub repo:
**https://github.com/srivathsan1510/redrob-india-runs-ranker**

This notebook clones that repo and runs `precompute.py` + `rank.py` on the 50-candidate sample file included in the repo (`data/sample_candidates.json`), which mirrors exactly what the real pipeline does on the full pool — just at a size that's fast and easy to inspect interactively.

## 1. Clone the repo and install dependencies

In [1]:
!git clone https://github.com/srivathsan1510/redrob-india-runs-ranker.git
%cd redrob-india-runs-ranker
!pip install -q -r requirements.txt

Cloning into 'redrob-india-runs-ranker'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 29 (delta 2), reused 28 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 17.15 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/redrob-india-runs-ranker


## 2. Phase A — Precompute (features + honeypot checks + embeddings)

On the real 100K-candidate pool this step has no time limit and is run once, locally. Here it runs against the included 50-candidate sample, which takes well under a minute including the one-time sentence-transformer model download.

In [2]:
!python precompute.py \
  --candidates data/sample_candidates.json \
  --jd-text-file config/jd_text.txt \
  --out-dir artifacts_demo \
  --embedder sentence-transformers

Loading candidates from data/sample_candidates.json ...
Loaded 50 candidates.
Extracting structured features ...
Saved structured features -> artifacts_demo/features.parquet (50 rows, 37 cols)
Computing embeddings with backend=sentence-transformers ...
modules.json: 100% 349/349 [00:00<00:00, 1.45MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 431kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 24.4MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 215kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.83MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 82.3MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 5408.96it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.56MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.73MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 33.8MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 505kB/s]
config.json: 100% 190/190 [00:00<00:00, 839kB/s]
Batches: 100% 1/1 [00:07<00:00,  7.61s/it]
Saved embeddings -> art

## 3. Phase B — Rank (the timed step: 5 min / 16GB / CPU / no network on the real 100K pool)

This is pure vectorized arithmetic over the artifacts from step 2 — no embedding computation happens here, which is why it stays fast and network-free regardless of pool size.

In [3]:
!python rank.py \
  --candidates data/sample_candidates.json \
  --artifacts-dir artifacts_demo \
  --out outputs/demo_submission.csv \
  --top-n 50

Loading precomputed artifacts ...
Loaded 50 candidates, embedding shape (50, 384).
Computing semantic JD-fit scores ...
Scoring structured components ...
Generating reasoning text ...
Wrote 50 ranked rows -> outputs/demo_submission.csv


## 4. Inspect the ranked output

Top of the list should show genuinely relevant titles (ML/search/ranking engineers); irrelevant titles (e.g. Accountant, Graphic Designer, Marketing Manager — present in the sample pool as distractors) should sink toward the bottom.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 150)

df = pd.read_csv('outputs/demo_submission.csv')
print(f"Ranked {len(df)} candidates.\n")
print("=== Top 10 ===")
display(df.head(10))

Ranked 50 candidates.

=== Top 10 ===


,candidate_id,rank,score,reasoning
0,CAND_0000031,1,0.4170,"Strong fit: 6.0 years of experience, working as Recommendation Systems Engineer at Swiggy, demonstrated background in ranking and information retr..."
1,CAND_0000001,2,0.1519,"High-confidence match: 6.9 years of experience, working as Backend Engineer at Mindtree, demonstrated background in vector database / hybrid searc..."
2,CAND_0000038,3,0.1302,"High-confidence match: 6.7 years in the field, working as Java Developer at Swiggy, demonstrated background in vector database / hybrid search inf..."
3,CAND_0000010,4,0.1144,"Strong fit: 4.6 years of experience, working as Data Engineer at Ola, demonstrated background in vector database / hybrid search infrastructure an..."
4,CAND_0000035,5,0.0925,"Solid match: 4.3 years of experience, working as Full Stack Developer at Globex Inc, demonstrated background in ranking and information retrieval,..."
5,CAND_0000021,6,0.0786,"Good fit overall: 14.5 years of experience, currently Project Manager at Wipro, demonstrated background in embeddings-based retrieval and vector d..."
6,CAND_0000002,7,0.0739,"Strong fit: 12.5 years of experience, currently Operations Manager at Wipro. Minor note: inactive on the platform for roughly 220 days."
7,CAND_0000015,8,0.0704,"Solid match: 5.4 years of experience, working as Software Engineer at Razorpay, demonstrated background in vector database / hybrid search infrast..."
8,CAND_0000011,9,0.0630,"Solid match: 2.0 years of experience, working as QA Engineer at Pied Piper, demonstrated background in ranking and information retrieval, with nam..."
9,CAND_0000032,10,0.0628,"Solid match: 8.1 years of experience, currently .NET Developer at Cognizant, demonstrated background in embeddings-based retrieval and Python, wit..."


In [5]:
print("=== Bottom 10 ===")
display(df.tail(10))

=== Bottom 10 ===


,candidate_id,rank,score,reasoning
40,CAND_0000005,41,0.0181,"Reasonable fit: 11.0 years of experience, working as Accountant at Stark Industries. Some concern: work history is closed-source with little exter..."
41,CAND_0000008,42,0.0169,"Reasonable fit: 3.6 years in the field, working as Operations Manager at Wipro. Some concern: career has been entirely at consulting firms with no..."
42,CAND_0000006,43,0.0159,"Reasonable fit: 6.0 years of experience, currently Business Analyst at Wayne Enterprises. Some concern: work history is closed-source with little ..."
43,CAND_0000003,44,0.0155,"Reasonable fit: 1.1 years of experience, currently Customer Support at TCS. Some concern: career has been entirely at consulting firms with no pro..."
44,CAND_0000009,45,0.0145,"Good fit overall: 11.0 years in the field, working as Mechanical Engineer at Dunder Mifflin. Some concern: work history is closed-source with litt..."
45,CAND_0000017,46,0.0135,"Good fit overall: 12.3 years in the field, working as Accountant at Wipro. Some concern: work history is closed-source with little external valida..."
46,CAND_0000037,47,0.0131,"Reasonable fit: 14.3 years of experience, currently Business Analyst at Stark Industries. Some concern: work history is closed-source with little ..."
47,CAND_0000045,48,0.0104,"Good fit overall: 12.2 years of experience, working as Project Manager at Initech. Some concern: work history is closed-source with little externa..."
48,CAND_0000047,49,0.0087,"Reasonable fit: 2.4 years in the field, currently Project Manager at TCS. Some concern: career has been entirely at consulting firms with no produ..."
49,CAND_0000028,50,0.0084,"Reasonable fit: 1.1 years in the field, currently Operations Manager at Wipro. Some concern: career has been entirely at consulting firms with no ..."


## 5. Cross-reference rank against actual job titles

This confirms the ranker is genuinely separating relevant from irrelevant candidates, not just reproducing the input order.

In [6]:
import json

candidates = json.load(open('data/sample_candidates.json'))
title_lookup = {c['candidate_id']: c['profile']['current_title'] for c in candidates}

df['current_title'] = df['candidate_id'].map(title_lookup)
display(df[['rank', 'candidate_id', 'current_title', 'score']])

,rank,candidate_id,current_title,score
0,1,CAND_0000031,Recommendation Systems Engineer,0.4170
1,2,CAND_0000001,Backend Engineer,0.1519
2,3,CAND_0000038,Java Developer,0.1302
3,4,CAND_0000010,Data Engineer,0.1144
4,5,CAND_0000035,Full Stack Developer,0.0925
5,6,CAND_0000021,Project Manager,0.0786
6,7,CAND_0000002,Operations Manager,0.0739
7,8,CAND_0000015,Software Engineer,0.0704
8,9,CAND_0000011,QA Engineer,0.0630
9,10,CAND_0000032,.NET Developer,0.0628


## 6. Validate against the official format checker

In [7]:
# Note: the official validator expects exactly 100 rows (the real submission
# requirement). This demo only ranks the 50 sample candidates, so the row-count
# check will correctly report a mismatch here — that's expected for a sandbox
# demo and is NOT present in the real outputs/submission.csv in this repo,
# which was produced from the full 100K pool and validates cleanly.
!python validate_submission.py outputs/demo_submission.csv

Validation failed (2 issue(s)):

- After the header (row 1), there must be exactly 100 data rows (rows 2–101); found 50.
- Each rank 1–100 must appear exactly once; missing: [51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


## About the approach

Two-phase pipeline (precompute / rank) with a relevance-first, gated scoring formula: semantic JD-fit (sentence-transformer embeddings) and rule-based skill-cluster matching form the core relevance score, which experience-band and location fit can only *modulate*, not replace — so a candidate with strong experience but zero relevant skills can't outrank a genuine match. Skill-cluster matches grounded in actual career-history text count far more than matches found only in a self-reported skills list, which directly defeats keyword-stuffing. Multiplicative penalties apply for six rule-based disqualifiers, narrative-consistency violations, and honeypot-consistency checks. Reasoning text is template-based and fact-grounded — every claim traces to an actual field in the candidate's record — with tone calibrated to both rank position and penalty severity.

Full rubric: `config/jd_profile.yaml` in the repo. Full methodology: `submission_metadata.yaml`.